# Procurement RAG Evaluation

This notebook reads every PDF in `documents/`, asks Azure OpenAI a grounded procurement question, and evaluates the answer against the extracted source text.

The model must not invent requirements. Claims are checked against the PDF corpus, and the answer must include a final `Sources` section.

In [1]:
%pip install pypdf tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import os
import re
import json
import pandas as pd
from dotenv import load_dotenv
from pypdf import PdfReader
from openai import AzureOpenAI

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'documents').exists():
    PROJECT_DIR = Path(__file__).resolve().parent if '__file__' in globals() else PROJECT_DIR
DOCUMENTS_DIR = PROJECT_DIR / 'documents'
load_dotenv(PROJECT_DIR / '.env')
print(f'Project: {PROJECT_DIR}')
print(f'Documents folder: {DOCUMENTS_DIR}')

Project: d:\BBI Intern\Graduation_Project\Final
Documents folder: d:\BBI Intern\Graduation_Project\Final\documents


In [3]:
def extract_pdf_corpus(folder: Path) -> list[dict]:
    records = []
    for pdf_path in sorted(folder.glob('*.pdf')):
        reader = PdfReader(str(pdf_path))
        pages = [(page.extract_text() or '').strip() for page in reader.pages]
        text = '\n\n'.join(page for page in pages if page)
        records.append({
            'source': pdf_path.name,
            'pages': len(reader.pages),
            'text': text,
        })
    return records

corpus = extract_pdf_corpus(DOCUMENTS_DIR)
corpus_df = pd.DataFrame([{
    'source': item['source'],
    'pages': item['pages'],
    'characters': len(item['text']),
} for item in corpus])
display(corpus_df)
print(f'Loaded {len(corpus)} PDF files.')

,source,pages,characters
0,Bank_Letter.pdf,1,558
1,Company_Profile.pdf,1,793
2,Procurement_Policy.pdf,1,1134
3,Purchase_Request_Guidelines.pdf,1,983
4,Supplier_Code_of_Conduct.pdf,1,1035
5,Tax_Certificate.pdf,1,560
6,Vendor_Onboarding_Guide.pdf,1,999
7,Vendor_Registration_Form.pdf,1,704
8,Vendor_Trade_License.pdf,1,757


Loaded 9 PDF files.


In [4]:
def build_context(records: list[dict]) -> str:
    return '\n\n---\n\n'.join(
        f"[Source Document: {item['source']}]\n{item['text']}"
        for item in records
    )

context = build_context(corpus)
print(f'Context characters: {len(context):,}')
print(context[:1500])

Context characters: 7,972
[Source Document: Bank_Letter.pdf]
Bank Account Confirmation Letter
Document Type
Bank Letter
Synthetic
Yes
Bank Confirmation
Sample Bank confirms that ABC Technology Trading LLC maintains an active corporate account.
 Field
Value
Account Name
ABC Technology Trading LLC
Bank Name
Sample Bank
IBAN
AE070331234567890123456
Account Currency
AED
Branch
Business Bay Branch
Letter Date
2026-01-20
Use of Letter
This letter is issued at the request of the account holder for vendor registration purposes.
Synthetic training document prepared for BBI Internship Program. Not for external/client use.

---

[Source Document: Company_Profile.pdf]
Company Profile
Document Type
Company Profile
Synthetic
Yes
Company Overview
ABC Technology Trading LLC is a technology services provider specializing in IT services, software
implementation, cloud consulting, and managed services.
Company Activities
The company provides solution implementation, application support, data platform con

In [5]:
endpoint = os.getenv('FOUNDRY_ENDPOINT') or os.getenv('AZURE_OPENAI_ENDPOINT')
api_key = os.getenv('FOUNDRY_KEY') or os.getenv('AZURE_OPENAI_API_KEY')
deployment = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME', 'gpt-4o')
if not endpoint or not api_key:
    raise RuntimeError('Missing FOUNDRY_ENDPOINT/AZURE_OPENAI_ENDPOINT or FOUNDRY_KEY/AZURE_OPENAI_API_KEY in .env')

client = AzureOpenAI(
    api_key=api_key,
    api_version='2024-06-01',
    azure_endpoint=endpoint,
)

SYSTEM_PROMPT = '''You are a procurement assistant. Answer only from the provided PDF context.
Do not invent, assume, or add approval requirements. Interpret amount thresholds cumulatively:
when a purchase exceeds a higher threshold, it also satisfies the requirements for lower
thresholds. Therefore, a purchase above 100,000 AED must include the requirements for
amounts above 25,000 AED, including at least three quotations unless an approved waiver
exists, and must also obtain Procurement Committee approval. Strategic or sensitive purchases
require Procurement Head review regardless of amount. If the context does not support an
answer, say what is missing and recommend the next procurement action. Do not approve
requests yourself. At the end, add exactly one Sources section listing the source documents
used. Do not cite sources elsewhere in the answer.'''

def ask_model(question: str) -> str:
    response = client.chat.completions.create(
        model=deployment,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT + '\n\nPDF CONTEXT:\n' + context},
            {'role': 'user', 'content': question},
        ],
    )
    return response.choices[0].message.content or ''

## Test Question

This test checks the approval threshold, quotations, validation checks, strategic purchases, and source citations.

In [6]:
question = 'What is the approval process for a purchase request above 100,000 AED?'
answer = ask_model(question)
print(answer)

For a purchase request above 100,000 AED, the following approval process must be followed:

1. **At least three quotations** must be obtained unless an approved waiver exists.  
2. **Procurement Committee approval** is required.  
3. If the purchase is strategic or sensitive, it must also undergo **Procurement Head review** regardless of the amount.  
4. The procurement officer must validate the request for completeness, budget availability, required quotations, vendor eligibility, and approval level before processing.  

Sources: Procurement_Policy.pdf, Purchase_Request_Guidelines.pdf


In [7]:
def source_names_in_answer(answer: str) -> list[str]:
    return [item['source'] for item in corpus if item['source'].lower() in answer.lower()]

def evaluate_answer(answer: str) -> dict:
    text = answer.lower()
    policy_terms = {
        'committee approval': 'procurement committee' in text and ('approval' in text or 'approve' in text),
        'three quotations': ('three' in text or '3' in text) and 'quotation' in text,
        'strategic/sensitive review': 'strategic' in text and ('sensitive' in text or 'procurement head' in text),
        'validation checks': any(term in text for term in ['budget', 'vendor eligibility', 'completeness']),
        'sources section': bool(re.search(r'(?im)^\s*sources\s*:?', answer)),
    }
    source_names = source_names_in_answer(answer)
    unsupported_claims = []
    if 'senior management' in text or 'board of directors' in text or 'executive committee' in text:
        unsupported_claims.append('Mentions management/board approval not stated in the provided policy.')
    score = round(100 * sum(policy_terms.values()) / len(policy_terms))
    return {
        'groundedness_score': score,
        'unsupported_claims': unsupported_claims,
        'source_documents_detected': ', '.join(source_names) or 'None',
        **policy_terms,
    }

evaluation = evaluate_answer(answer)
display(pd.DataFrame([evaluation]).T.rename(columns={0: 'value'}))

,value
groundedness_score,100
unsupported_claims,[]
source_documents_detected,"Procurement_Policy.pdf, Purchase_Request_Guide..."
committee approval,True
three quotations,True
strategic/sensitive review,True
validation checks,True
sources section,True


## Interpretation

A high score does not replace human review. Inspect every unsupported claim and confirm that each source listed by the model actually supports the answer. For stronger evaluation, add more questions covering vendor eligibility, missing purchase-request fields, quotation waivers, and validation by request ID.

In [8]:
%pip install pypdf tabulate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Batch Evaluation: Relevance and Groundedness

The two request-specific questions are intentionally excluded:
- `What information is missing from purchase request PR-2026-0005?`
- `Validate whether PR-2026-0010 is ready for processing.`

The remaining questions are evaluated using only two metrics:
- **Relevance:** Does the answer directly address the user's question?
- **Groundedness:** Are the answer's claims supported by the PDF context?

In [9]:
test_cases = [
    {
        'question': 'What documents are required to onboard a new vendor?',
        'expected_sources': ['Vendor_Onboarding_Guide.pdf'],
    },
    {
        'question': 'What is the approval process for a purchase request above 100,000 AED?',
        'expected_sources': ['Procurement_Policy.pdf', 'Purchase_Request_Guidelines.pdf'],
    },
    {
        'question': 'Can a vendor be approved if the trade license is expired?',
        'expected_sources': ['Procurement_Policy.pdf', 'Vendor_Onboarding_Guide.pdf'],
    },
    {
        'question': 'Summarize ABC Technology Trading LLC vendor profile.',
        'expected_sources': ['Company_Profile.pdf'],
    },
    {
        'question': 'Does a purchase request for 80,000 AED require three quotations?',
        'expected_sources': ['Purchase_Request_Guidelines.pdf'],
    },
    {
        'question': 'What are the main compliance requirements in the supplier code of conduct?',
        'expected_sources': ['Supplier_Code_of_Conduct.pdf'],
    },
    {
        'question': 'Which fields should be extracted from the vendor trade license?',
        'expected_sources': ['Vendor_Trade_License.pdf'],
    },
    {
        'question': 'What should procurement do if budget code is missing?',
        'expected_sources': ['Purchase_Request_Guidelines.pdf'],
    },
    {
        'question': 'Generate a short procurement officer summary for ABC Technology Trading LLC.',
        'expected_sources': ['Vendor_Trade_License.pdf', 'Tax_Certificate.pdf', 'Bank_Letter.pdf', 'Vendor_Registration_Form.pdf', 'Company_Profile.pdf'],
    },
    {
        'question': 'What risks should be considered before approving a vendor?',
        'expected_sources': ['Procurement_Policy.pdf', 'Vendor_Onboarding_Guide.pdf', 'Supplier_Code_of_Conduct.pdf'],
    },
]

JUDGE_PROMPT = '''You are evaluating a procurement assistant answer against the provided PDF context.
Return valid JSON only with this exact shape:
{"relevance_score": 0, "groundedness_score": 0, "missing_key_points": [], "reason": "..."}
Use scores from 0 to 100.
Relevance measures whether the answer directly addresses the user's question.
Groundedness measures whether the answer is supported by the PDF context and does not invent facts.
Do not score based on writing style. List important omissions in missing_key_points.
'''

def judge_answer(question: str, answer: str) -> dict:
    response = client.chat.completions.create(
        model=deployment,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': JUDGE_PROMPT + '\n\nPDF CONTEXT:\n' + context},
            {'role': 'user', 'content': f'Question: {question}\n\nAnswer:\n{answer}'},
        ],
    )
    try:
        return json.loads(response.choices[0].message.content or '{}')
    except json.JSONDecodeError:
        return {
            'relevance_score': 0,
            'groundedness_score': 0,
            'missing_key_points': ['Judge returned invalid JSON.'],
            'reason': 'Evaluation could not be parsed.',
        }

def run_batch_evaluation(cases: list[dict]) -> pd.DataFrame:
    rows = []
    for case in cases:
        answer = ask_model(case['question'])
        judgment = judge_answer(case['question'], answer)
        detected_sources = source_names_in_answer(answer)
        rows.append({
            'question': case['question'],
            'relevance_score': float(judgment.get('relevance_score', 0)),
            'groundedness_score': float(judgment.get('groundedness_score', 0)),
            'missing_key_points': '; '.join(judgment.get('missing_key_points', [])),
            'expected_sources': ', '.join(case['expected_sources']),
            'detected_sources': ', '.join(detected_sources) or 'None',
            'answer': answer,
        })
    return pd.DataFrame(rows)

batch_results = run_batch_evaluation(test_cases)
summary = pd.DataFrame([{
    'questions_evaluated': len(batch_results),
    'average_relevance': round(batch_results['relevance_score'].mean(), 1),
    'average_groundedness': round(batch_results['groundedness_score'].mean(), 1),
}])
display(summary)
display(batch_results[['question', 'relevance_score', 'groundedness_score', 'missing_key_points', 'detected_sources']])

,questions_evaluated,average_relevance,average_groundedness
0,10,98.5,98.5


,question,relevance_score,groundedness_score,missing_key_points,detected_sources
0,What documents are required to onboard a new v...,100.0,100.0,,Vendor_Onboarding_Guide.pdf
1,What is the approval process for a purchase re...,100.0,100.0,,"Procurement_Policy.pdf, Purchase_Request_Guide..."
2,Can a vendor be approved if the trade license ...,100.0,100.0,,"Procurement_Policy.pdf, Vendor_Onboarding_Guid..."
3,Summarize ABC Technology Trading LLC vendor pr...,95.0,90.0,The summary does not explicitly mention the co...,"Bank_Letter.pdf, Company_Profile.pdf, Tax_Cert..."
4,"Does a purchase request for 80,000 AED require...",100.0,100.0,,"Procurement_Policy.pdf, Purchase_Request_Guide..."
5,What are the main compliance requirements in t...,100.0,100.0,,Supplier_Code_of_Conduct.pdf
6,Which fields should be extracted from the vend...,100.0,100.0,,"Vendor_Onboarding_Guide.pdf, Vendor_Trade_Lice..."
7,What should procurement do if budget code is m...,100.0,100.0,,Purchase_Request_Guidelines.pdf
8,Generate a short procurement officer summary f...,100.0,100.0,,"Bank_Letter.pdf, Company_Profile.pdf, Procurem..."
9,What risks should be considered before approvi...,90.0,95.0,Specific examples of compliance risks that cou...,"Procurement_Policy.pdf, Supplier_Code_of_Condu..."
